<a href="https://colab.research.google.com/github/hmmnyamminji/python/blob/main/%EB%AC%B8%EC%9E%90%EC%97%B4_%EB%B2%94%EC%A3%BC%ED%98%95%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 환경 설정

In [ ]:
# User's request: "Title Other 말고" (Not 'Other' for Title).
# To refine the 'Other' category in the 'Title' column, you would typically:
# 1. Inspect the unique titles that are currently being grouped into 'Other'.
#    (e.g., by checking `titanic['Name'].str.extract(r'.Ws([^W.]+)W.').squeeze().value_counts()` in the relevant cell).
# 2. Based on the unique titles, define a more granular categorization.
#
# Example of how to refine title categorization (to be applied in the cell where 'Title' is created, e.g., cell dNc31ehP0DCX):
# import re # Ensure 're' is imported if not already.
# def refine_title(title):
#     if title in ['Mr', 'Miss', 'Mrs', 'Master']:
#         return title
#     elif title == 'Dr':
#         return 'Doctor'
#     elif title in ['Rev', 'Col', 'Capt']:
#         return 'Officer'
#     elif title == 'Mlle': # Mademoiselle
#         return 'Miss'
#     elif title == 'Mme': # Madame
#         return 'Mrs'
#     else:
#         return 'Rare'
#
# # Then, you would apply this function to your DataFrame's raw extracted titles:
# # titanic['Title'] = titanic['Name'].str.extract(r'.Ws([^W.]+)W.').squeeze().apply(refine_title)

# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 드라이브에 저장된 폰트 등록
import matplotlib as mpl
import matplotlib.pyplot as plt # 그래프를 그리는 pyplot 모듈
import matplotlib.font_manager as fm # 폰트 관리 모듈

# 드라이브 내 폰트 경로
font_path = '/content/drive/MyDrive/kwu/Bigdata/dataPreProcessing/NanumGothic.ttf'

fm.fontManager.addfont(font_path)
mpl.rc('font',family='NanumGothic') #matplotlib 기본 폰트로 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.sans-serif'] = ['NanumGothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 기호가 깨지지 않도록 비활성화

print("현재 폰트: ", plt.rcParams['font.family'])

Mounted at /content/drive
현재 폰트:  ['NanumGothic']


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import warnings # 코드 실행 중 발생하는 불피요한 경고 메시지를 숨김
warnings.filterwarnings('ignore')
import pprint

In [ ]:
import pandas as pd # 데이터프레임 조작
import numpy as np # 수치 계산 및 NaN 처리
import matplotlib.pyplot as plt # 그래프 시각화
import seaborn as sns # 통계 시각화

# 타이타닉 데이터 로드
titanic = pd.read_csv('/content/drive/MyDrive/kwu/Bigdata/dataPreProcessing/train.csv')
print(titanic.shape)
display(titanic.head())
titanic.head()


(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
# LabelEncoder : 범주형 문자열을 정수로 변환, 순서형에서 사용
# OneHotEncoder : 순서가 없는 명목형 변수에 사용
# OrdinalEncoder : 순서가 있는 범주형 변수에 사용

In [ ]:
# 결측치 처리
titanic['Age'] = titanic.groupby(['Sex','Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
titanic = titanic.drop(columns=['Cabin'], errors='ignore')
titanic['Title'] = titanic['Name'].str.extract(r'.\s([^\.]+)\.').squeeze()                                                #DataFrame -> Series로 변환
titanic['Title'] = titanic['Title'].apply(
    lambda x: x if x in ['Mr', 'Miss', 'Mrs', 'Master'] else 'Other'
)
titanic['AgeGroup'] = pd.cut(
    titanic['Age'], bins=[0, 12, 17, 59, 100], labels=['아동','청소년','성인','노인']
)

titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1

print(titanic[['Sex','Embarked','Pclass','AgeGroup','Title']].head())

      Sex Embarked  Pclass AgeGroup Title
0    male        S       3       성인    Mr
1  female        C       1       성인   Mrs
2  female        S       3       성인  Miss
3  female        S       1       성인   Mrs
4    male        S       3       성인    Mr


### str 접근자로 문자열 정체

In [ ]:
# 공백, 대소문자 처리
sample = pd.Series(['Male','Female','MALE','Female'])
print('원본:', sample.tolist())
print('strip:', sample.str.strip().tolist())
print('lower:', sample.str.lower().tolist())
print('strip+lower:', sample.str.strip().str.lower().tolist())

원본: ['Male', 'Female', 'MALE', 'Female']
strip: ['Male', 'Female', 'MALE', 'Female']
lower: ['male', 'female', 'male', 'female']
strip+lower: ['male', 'female', 'male', 'female']


### str.contains() / str.extract()로 패턴 탐색

In [ ]:
has_mr = titanic['Name'].str.contains('Mr\.') # 포함 여부

has_miss = titanic['Name'].str.contains('Miss\.')
print(f"Mr, 호칭:{has_mr.sum()}명")
print(f"Miss, 호칭:{has_miss.sum()}명")

Mr, 호칭:517명
Miss, 호칭:182명


In [ ]:
titanic['TicketType'] = titanic['Ticket'].str.match(r'^\d+$').map(
    {True: '숫자형', False: '문자포함형'}
)
print("\n티켓 유형별 생존율:")
print(titanic.groupby('TicketType')['Survived'].mean().round(3))



티켓 유형별 생존율:
TicketType
문자포함형    0.383
숫자형      0.384
Name: Survived, dtype: float64


### 레이블 인코딩(순서형) - map() / replace()

In [ ]:
# map()으로 직접 매핑
titanic['Sex_encoded'] = titanic['Sex'].map({'male': 0, 'female': 1})
print("성별 인코딩: ")
print(titanic[['Sex','Sex_encoded']].value_counts().reset_index())

# replace()로 치환, 원본 값 유지 옵션 있음
titanic['Embarked_encoded'] = titanic['Embarked'].replace({'S': 0, 'C':1, 'Q':2})
titanic[['Embarked', 'Embarked_encoded']].head()

성별 인코딩: 
      Sex  Sex_encoded  count
0    male            0    577
1  female            1    314


,Embarked,Embarked_encoded
0,S,0
1,C,1
2,S,0
3,S,0
4,S,0


### 원-핫 인코딩 - pd.get_dummies()

In [ ]:
embarked_dummies = pd.get_dummies(titanic['Embarked'], prefix='embarked')
print("원-핫 인코딩 결과: ")
print(embarked_dummies.head())

원-핫 인코딩 결과: 
   embarked_C  embarked_Q  embarked_S
0       False       False        True
1        True       False       False
2       False       False        True
3       False       False        True
4       False       False        True


In [ ]:
embarked_dummies = pd.get_dummies(titanic['Embarked'], prefix='embarked', drop_first=True)
print("원-핫 인코딩 결과: ")
print(embarked_dummies.head()) # 알파벳 첫 번째 변주(C) 열 제거, 나머지로 추론 가능

원-핫 인코딩 결과: 
   embarked_Q  embarked_S
0       False        True
1       False       False
2       False        True
3       False        True
4       False        True


In [ ]:
# 여러 열 한번에 인코딩
cols_to_encode = ['Sex','Embarked','Title']
titanic_ohe = pd.get_dummies(titanic, columns=cols_to_encode, drop_first=True)
print("원-핫 인코딩 결과: ")
print(titanic_ohe.head())

원-핫 인코딩 결과: 
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare AgeGroup  ...  TicketType Sex_encoded  \
0         A/5 21171   7.2500       성인  ...       문자포함형           0   
1          PC 17599  71.2833       성인  ...       문자포함형           1   
2  STON/O2. 3101282   7.9250       성인  ...       문자포함형           1   
3            113803  53.1000     

### 원-핫 인코딩 - sklearn OneHotEncoder

In [ ]:
# sklearn OneHotEncoder: 훈련/테스트 분리 시 유용 (새 범주 대응 가능)
from sklearn.model_selection import train_test_split

X = titanic[['Sex','Embarked']].copy()
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# fit()은 훈련 데이터로만, 훈련 데이터의 범주 목록 학습
ohe = OneHotEncoder(
    sparse_output=False, #결과를 밀집 배열(numpy array)로 변환
    drop='first', # 각 범주의 첫 번째 열 제거
    handle_unknown='ignore' # 새 범주가 테스트에 등장해도 오류 없이 모두 0으로 처리
)
ohe.fit(X_train) # 훈련 데이터의 범주 목록 학습, 이 정보를 기준으로 이후 transform() 수행

# transform()
X_train_ohe = ohe.transform(X_train) #훈련 데이터 학습된 범주 기준으로 원-핫 인코딩
X_test_ohe = ohe.transform(X_test) # 테스트 데이터도 훈련 데이터 기준 범주로 변환

print('생성된 열:', ohe.get_feature_names_out()) # 인코딩 후 생성된 열 이름 배열 출력
print("훈련 데이터 shape:", X_train_ohe.shape)
print("\n결과 예시 (첫 3행):")
result_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out())
print(result_df.head(3))

생성된 열: ['Sex_male' 'Embarked_Q' 'Embarked_S']
훈련 데이터 shape: (712, 3)

결과 예시 (첫 3행):
   Sex_male  Embarked_Q  Embarked_S
0       1.0         0.0         1.0
1       1.0         0.0         1.0
2       1.0         0.0         1.0


### 순서형 인코딩 - 직접 매핑

In [ ]:
# 순서형(ordinal) 인코딩: 범주 간에 의미 있는 순서가 있을 때 순서를 반영해서 점수 부여
titanic['Pclass_encoded'] = titanic['Pclass'].map({1:2, 2:1, 3:0})
print("객실 등급 순서형 인코딩:")
print(titanic[['Pclass','Pclass_encoded']].drop_duplicates().sort_values('Pclass'))

age_order = {'아동': 0, '청소년': 1, '성인': 2, '노인': 3}
titanic['AgeGroup_encoded'] = titanic['AgeGroup'].map(age_order)
print("\n나이 구간 순서형 인코딩:")
print(titanic[['AgeGroup','AgeGroup_encoded']].drop_duplicates().sort_values('AgeGroup_encoded').to_string())

객실 등급 순서형 인코딩:
   Pclass  Pclass_encoded
1       1               2
9       2               1
0       3               0

나이 구간 순서형 인코딩:
   AgeGroup AgeGroup_encoded
7        아동                0
9       청소년                1
0        성인                2
33       노인                3


### 순서형 인코딩 - sklearn OrdinalEncoder

In [ ]:
# OrdinalEncoder는 2D 입력 필요하므로 [[]] 사용
oe = OrdinalEncoder(
    categories=[['아동','청소년','성인','노인']],
    handle_unknown='use_encoded_value', # 학습 시 없었던 새 범주가 등장하면 오류 대신 unknown_value로 처리
    unknown_value=-1 # 새로운 범주에 부여할 값

)

age_group_col = titanic[['AgeGroup']].astype(str)
titanic['AgeGroup_ordinal']= oe.fit_transform(age_group_col)
print(titanic[['AgeGroup', 'AgeGroup_ordinal']].drop_duplicates().sort_values('AgeGroup_ordinal').to_string())

   AgeGroup  AgeGroup_ordinal
7        아동               0.0
9       청소년               1.0
0        성인               2.0
33       노인               3.0


### 빈도 인코딩 & 타깃 인코딩 (고급)

In [ ]:
# 빈도 인코딩: 각 범주의 등장 횟수(빈도)로 대체
freq_map = titanic['Title'].value_counts().to_dict()
titanic['Title_freq'] = titanic['Title'].map(freq_map)
print("빈도 인코딩 결과:")
print(titanic[['Title', 'Title_freq']].drop_duplicates()
      .sort_values('Title_freq', ascending=False))

# 타깃 인코딩: 각 범주의 목표 변수(생존율) 평균으로 대체
target_map = titanic.groupby('Title')['Survived'].mean().to_dict()
titanic['Title_target'] = titanic['Title'].map(target_map)
print("\n타깃 인코딩 결과(생존율 기반):")
print(titanic[['Title', 'Title_target']].drop_duplicates()
      .sort_values('Title_target', ascending=False).round(3))

빈도 인코딩 결과:
     Title  Title_freq
0       Mr         502
2     Miss         179
1      Mrs         121
18   Other          49
7   Master          40

타깃 인코딩 결과(생존율 기반):
     Title  Title_target
1      Mrs         0.802
2     Miss         0.704
7   Master         0.575
18   Other         0.347
0       Mr         0.157


### 타이타닉 전처리

In [ ]:
titanic_final = pd.read_csv('/content/drive/MyDrive/kwu/Bigdata/dataPreProcessing/train.csv')
titanic_final = titanic_final.drop(columns=['Cabin'])

# 결측치 처리
titanic_final['Age'] = titanic_final.groupby(['Sex','Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
titanic_final['Embarked'] = titanic_final['Embarked'].fillna(
    titanic_final['Embarked'].mode()[0]
)
# 이상치 처리 - 캡핑
Q1, Q3 = titanic_final['Fare'].quantile([0.25, 0.75])
titanic_final['Fare'] = titanic_final['Fare'].clip(upper=Q3+1.5*(Q3-Q1)) #초과 값을 상한으로 대체

# 분포 변환 - 로그 변환
titanic_final['Fare'] = np.log1p(titanic_final['Fare']) # 양의 왜도

# 파생 변수 생성
titanic_final['FamilySize'] = titanic_final['SibSp'] + titanic_final['Parch'] + 1
titanic_final['IsAlone'] = (titanic_final['FamilySize'] == 1).astype(int)
titanic_final['Title'] = titanic_final['Name'].str.extract(r'.\s*([^\.]+)\.').squeeze()
titanic_final['Title'] = titanic_final['Title'].apply(
    lambda x: x if x in ['Mr', 'Miss','Mrs', 'Master'] else 'Other'
)
# 범주형 인코딩
# 명목형 변수(순서 없음) -> 원-핫 인코딩
titanic_final = pd.get_dummies(titanic_final, columns=['Sex','Embarked','Title'], drop_first=True)

# 순서형 변수(순서 있음) -> 순서형 인코딩
titanic_final['Pclass'] = titanic_final['Pclass'].map({1: 2, 2: 1, 3: 0})

# 불필요 열 제거
titanic_final = titanic_final.drop(columns=['Name','Ticket','SibSp','Parch'])

# 스케일링
from sklearn.preprocessing import StandardScaler # 평균=0, 표준편차=1로 표준화
num_cols = ['Age','Fare','FamilySize']
scaler = StandardScaler()
titanic_final[num_cols] = scaler.fit_transform(titanic_final[num_cols])

# 최종 결과 확인
print(f"\n최종 shape:{titanic_final.shape}")
print(f"결측치: {titanic_final.isnull().sum().sum()}개")
print(f"\n최종 컬럼 목록:")
print(titanic_final.columns.tolist())
print(f"\n데이터 미리보기:")
print(titanic_final.head(3).round(3))


최종 shape:(891, 10)
결측치: 0개

최종 컬럼 목록:
['PassengerId', 'Survived', 'Pclass', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'Sex_male', 'Embarked_Q', 'Embarked_S']

데이터 미리보기:
   PassengerId  Survived  Pclass    Age   Fare  FamilySize  IsAlone  Sex_male  \
0            1         0       0 -0.535 -0.938       0.059        0      True   
1            2         1       2  0.668  1.563       0.059        0     False   
2            3         1       0 -0.234 -0.844      -0.561        1     False   

   Embarked_Q  Embarked_S  
0       False        True  
1       False       False  
2       False        True  
